# D71 冲刺 A —— 移动信任域(`ppo_push_A_s0`)

init=D70 best(182,745), **锚也=同一 best**(围着最好的自己小步爬), β 0.01, lr 5e-5, 旧特征(fv=1)。与 AB 版唯一差异 = `--feature-version 1`。
开头应打印: `[init] ... 载入 N 层原样` + `[anchor] ... 移动信任域`。

## 设置

In [ ]:
import sys, os
venv_site = '/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/lib/python3.11/site-packages'
if os.path.isdir(venv_site) and venv_site not in sys.path:
    sys.path.insert(0, venv_site)
%cd /rds/homes/h/hxn886/ondemand/final_project
PROJ='/rds/homes/h/hxn886/ondemand/final_project'
PY='/rds/homes/h/hxn886/virtual-environments/my-virtual-env-sapphire/bin/python'
PREFIX=(f'cd {PROJ} && env PYTHONUNBUFFERED=1 PYTHONPATH=. '
        f'PYTORCH_CUDA_ALLOC_CONF=expandable_segments:True MPLBACKEND=Agg {PY}')
print('OK')


## 训练(重跑=续训)
**D73 死锁圈手术已修**: 09-28 的恒定 2000 万噪声消除, 档3 现可通关 → 升档4(全高峰)通道打开。直接重跑本格(resume 从 it99 续, --iters 已加到 160), 观察: 档3 train 延误应从 2000 万掉到正常值、随后升档4。


In [ ]:
!{PREFIX} scripts/13_train_ppo.py \
    --run-tag ppo_push_A_s0 --feature-version 1 \
    --init-ckpt artifacts/reports/ppo_8192_anchor_s0_best.pt --anchor-ckpt artifacts/reports/ppo_8192_anchor_s0_best.pt \
    --bc-anchor-beta 0.01 --lr 5e-5 \
    --encoder gnn --control-classes 12 \
    --kl-anchor schedule --kl-beta 0.03 --seed 0 --reserve-window wide \
    --curriculum --curriculum-tiers 185,286,300,999999 \
    --promote-at 1.0 --promote-delay-k 1.0 \
    --train-eval-days 6 --iters 160 --steps-per-iter 8192 --eval-every 5 \
    --early-stop-patience 8 --best-rescue --resume


## 对比(两份都有结果后在任一 notebook 跑)

In [ ]:
import json, torch
from pathlib import Path
rep=Path('artifacts/reports')
def load(tag):
    if (rep/f'{tag}.json').exists(): return json.loads((rep/f'{tag}.json').read_text())
    if (rep/f'{tag}_last.pt').exists(): return torch.load(rep/f'{tag}_last.pt',map_location='cpu',weights_only=False)
    return None
runs=[('push A(纯A)','ppo_push_A_s0'),('push AB(A+前瞻)','ppo_push_AB_s0'),('D70 锚定基线','ppo_8192_anchor_s0')]
loaded={n:load(t) for n,t in runs}
base=next((d.get('baselines') for d in loaded.values() if d and d.get('baselines')),{})
print(f"{'run':<16}{'best完成':>9}{'best延误':>11}{'best_rew':>10}{'@it':>5}")
for k in ['fcfs','lookup_most_frequent','greedy_set','signaller_replay']:
    if k in base:
        b=base[k]; print(f"{k:<16}{b['eval_completion_rate']:>9.3f}{b['eval_weighted_secondary_delay_s']:>11.0f}{b['eval_episode_reward']:>10.0f}")
for name,_ in runs:
    d=loaded.get(name)
    if not d or not d.get('best'): print(f'{name:<16}(missing)'); continue
    b=d['best']; print(f"{name:<16}{b.get('completion',0):>9.3f}{b.get('delay',0):>11.0f}{b.get('reward',0):>10.0f}{b.get('iter',-1):>5}")
print('\n靶: 延误 < fcfs 181164 / reward > 15918 = 超过全部规则基线')
try:
    import matplotlib; matplotlib.use('Agg'); import matplotlib.pyplot as plt
    plt.figure(figsize=(8,3.2))
    for name,_ in runs:
        d=loaded.get(name)
        if d and d.get('evals'):
            ev=d['evals']; plt.plot([e['iter'] for e in ev],[e['eval_weighted_secondary_delay_s'] for e in ev],marker='.',label=name)
    if 'fcfs' in base: plt.axhline(base['fcfs']['eval_weighted_secondary_delay_s'],ls='--',c='b',lw=.9,label='fcfs 181k')
    plt.ylim(150000, 300000); plt.ylabel('val 延误'); plt.xlabel('iter'); plt.legend(fontsize=8)
    plt.title('冲刺超 fcfs(D71)'); plt.tight_layout(); plt.savefig('artifacts/reports/d71_push.png',dpi=110)
    print('fig -> artifacts/reports/d71_push.png')
except Exception as e: print('plot skip',e)
